In [1]:
import pandas as pd

In [49]:
# Load data
df = pd.read_pickle("../data/agent_df_base_res_national_revised_v2.pkl")
load_growth = pd.read_csv("../data/Experimental_load_growth.csv")

In [51]:
df.columns

Index(['pgid', 'tract_id_alias', 'bin_id', 'bldg_id', 'state_abbr',
       'census_division_abbr', 'reeds_reg', 'customers_in_bin_initial',
       'load_kwh_per_customer_in_bin_initial', 'developable_roof_sqft',
       'load_kwh_in_bin_initial', 'max_demand_kw', 'avg_monthly_kwh',
       'crb_model', 'hdf_load_index', 'owner_occupancy_status',
       'cap_cost_multiplier', 'solar_re_9809_gid', 'tilt', 'azimuth',
       'pct_of_bldgs_developable', 'bldg_size_class', 'i', 'j', 'cf_bin',
       'sector_abbr', 'sector', 'tech', 'ba', 'rto', 'roof_adjustment',
       'county_id', 'tariff_name', 'tariff_dict', 'tariff_id', 'eia_id'],
      dtype='object')

In [52]:
df.loc[df['state_abbr']=='WI']['customers_in_bin_initial'].unique()

array([5929.57766131528, 3227.6779506204, 3320.59564617615,
       3699.7402430417, 4939.98948372067, 1213.58590195428,
       2616.44110985581, 1949.45365175777, 4663.25634695679,
       2155.48854190314, 3113.34878608876, 5656.46571445166,
       2212.24913418829, 1358.41631002705, 3054.56824390023,
       3289.48841766401, 735.8677498035, 1837.54842711019,
       2666.13187747911, 2672.3937221796, 4818.79248951751,
       5342.16150948481, 2159.12445172924, 3036.59068976009,
       471.052317469602, 5175.11165247479, 3029.52086509824,
       3556.72778988197, 2002.9823241975, 4103.93221870922,
       4202.7077689848, 2519.48351449329, 3441.59064538897,
       4842.22390839679, 1899.3588941538, 5174.1016775231,
       4961.40095269656, 971.393908538301, 1846.43620668509,
       1999.34641437141, 1802.60329378162, 1468.70557475192,
       4440.05188263264, 2821.26403005915, 2185.78779045393,
       1803.81526372365, 3991.01701910995], dtype=object)

In [60]:
# 1) Collapse SF attached + detached into one “Single-Family” group
df = df.copy()
df['crb_model_group'] = df['crb_model'].replace({
    'Single-Family Detached': 'Single-Family',
    'Single-Family Attached':  'Single-Family'
})

# 2) Ensure dtypes
df['owner_occupancy_status'] = df['owner_occupancy_status'].astype(int)
df['customers_in_bin_initial'] = df['customers_in_bin_initial'].astype(int)
df['pct_of_bldgs_developable'] = df['pct_of_bldgs_developable'].astype(float)

# 3) Compute “can solar” counts
df['owner_can_solar'] = (
    df['customers_in_bin_initial']
    * df['pct_of_bldgs_developable']
    * (df['owner_occupancy_status'] == 1)
    * (df['crb_model_group'] != 'Multi-Family with 5+ Units')  # Exclude large multi-family
).round(0)

df['renter_can_solar'] = (
    df['customers_in_bin_initial']
    * df['pct_of_bldgs_developable']
    * (df['owner_occupancy_status'] == 2)
).round(0)

# 4) Create owner/renter totals
df['owners']  = df['customers_in_bin_initial'] * (df['owner_occupancy_status'] == 1)
df['renters'] = df['customers_in_bin_initial'] * (df['owner_occupancy_status'] == 2)
df['total_households'] = df['customers_in_bin_initial']

# 5) Group and aggregate, then compute proportions
result = (
    df
    .groupby(['state_abbr'], as_index=False)
    [['renters', 'owners', 'owner_can_solar', 'total_households']]
    .sum()
    .assign(
        pct_renters=lambda d: d['renters'] / d['total_households']*100,
        pct_owners =lambda d: d['owners'] / d['total_households']*100,
        pct_owners_can_solar=lambda d: d['owner_can_solar'] / d['total_households']*100,
    )[['state_abbr', 'prop_renters', 'prop_owners', 'prop_owners_can_solar']]
    .round(1) 
)

KeyError: "['prop_renters', 'prop_owners', 'prop_owners_can_solar'] not in index"

In [61]:
result

,state_abbr,prop_renters,prop_owners,prop_owners_can_solar
0,AL,26.9,73.1,60.2
1,AR,26.6,73.4,61.2
2,AZ,35.9,64.1,51.8
3,CA,44.7,55.3,47.0
4,CO,32.1,67.9,47.9
5,CT,36.4,63.6,45.8
6,DC,64.3,35.7,25.4
7,DE,36.2,63.8,49.8
8,FL,35.6,64.4,51.5
9,GA,36.2,63.8,50.9


In [62]:
result.to_csv('../data/technical_potential_analysis.csv', index=False)